# Step 3: training loop + toy-set overfit
`train_epoch` and `predict` live in `12_training_step.py` (`13_real_training.ipynb` imports them from there). This notebook checks that they work by overfitting a 10-pair toy set.

Done when: the toy set overfits to loss < 0.01, with all 10 pairs predicted correctly. That proves vocab → encoding → encoder → siamese head → loss → optimiser all connect and can learn. **Done** (loss 0.0003 by epoch 250, 10/10 correct).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from helpers.pairs import load_pairs

build_vocab = import_module("siamese_model.01_vocab_step").build_vocab          # digit-prefixed filenames -> can't use `from ... import`
_model_step = import_module("siamese_model.08_encoder_step")
Encoder, SiameseMatcher = _model_step.Encoder, _model_step.SiameseMatcher
_data_step = import_module("siamese_model.11_data_plumbing")
PairDataset, make_collate = _data_step.PairDataset, _data_step.make_collate
_train_step = import_module("siamese_model.12_training_step")
train_epoch, predict = _train_step.train_epoch, _train_step.predict


In [2]:
ot_train, ot_train_label = load_pairs("overture", "train")   # vocab from the real train split (the toy strings only use its characters)
vocab = build_vocab(ot_train)

## 1. `train_epoch` and `predict` (in `12_training_step.py`)
- **`train_epoch(model, loader, opt, loss_fn)`**: one pass over the loader, updating the weights. Returns the average loss per pair (each batch's loss × its size, ÷ total pairs). Loop order: `zero_grad` → forward → loss → `backward` → clip → `step`. Clipping sits between `backward` and `step` because it edits the gradients `step` is about to use.
- **`predict(model, loader)`**: `eval()` + `no_grad()`, no weight updates. Returns a 1-D tensor of probabilities in loader order, so val/test loaders must not shuffle. It applies `sigmoid` itself, because the model returns raw logits (`BCEWithLogitsLoss` applies the sigmoid during training).

## 2. Toy set
10 hand-written pairs: identical, abbreviation, typo and extra-unit positives; case1-style (same street, different number), case2-style (same number, different street) and easy negatives.

In [3]:
toy = pd.DataFrame([
    ("12 main st",        "12 main st",         1),   # identical
    ("12 main st",        "12 main street",     1),   # abbreviation
    ("45 oak avenue",     "45 oak ave",         1),   # abbreviation
    ("301 elm road",      "301 elm raod",       1),   # typo
    ("88 pine st apt 2",  "88 pine st",         1),   # extra unit (building-level match)
    ("12 main st",        "14 main st",         0),   # case1: same street, different number
    ("45 oak avenue",     "47 oak avenue",      0),   # case1
    ("301 elm road",      "301 elm street",     0),   # case2-style: same number, different street type
    ("12 main st",        "990 harbor view dr", 0),   # easy
    ("88 pine st",        "5 quarry lane",      0),   # easy
], columns=["address_a", "address_b", "label"])
toy_labels = toy["label"]

toy_ds = PairDataset(toy, toy_labels)
toy_loader = DataLoader(toy_ds, batch_size=len(toy_ds), shuffle=False, collate_fn=make_collate(vocab))   # one batch = the whole toy set

## 3. Overfit the toy set
Seed first (reproducible), then model → optimiser (needs the model's parameters) → loss function. 300 epochs, one batch each (the whole toy set). Tests directly below.

In [4]:
torch.manual_seed(0)
model = SiameseMatcher(Encoder(len(vocab)))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
for epoch in range(300):
    loss = train_epoch(model, toy_loader, opt, loss_fn)
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d} loss {loss:.4f}")

epoch   0 loss 0.6928
epoch  50 loss 0.0110
epoch 100 loss 0.0009
epoch 150 loss 0.0005
epoch 200 loss 0.0004
epoch 250 loss 0.0003


In [5]:
# Tests: the overfit run, then predict on the trained model (expected values written down before running)
assert loss < 0.01                                          # final loss near zero (last epoch's loss from the loop above)

predicted = predict(model, toy_loader)
assert predicted.shape == (10,)                             # one probability per pair
assert ((predicted >= 0) & (predicted <= 1)).all()          # probabilities, not logits
assert torch.equal(predict(model, toy_loader), predicted)   # predict doesn't train: a second call gives identical probabilities
assert (predicted.round() == torch.tensor(toy_labels)).all()   # all 10 toy pairs correct
print("all tests pass")

all tests pass
